# Where Does Matter Stop Existing?
### A quantified study of nuclear mass models and the neutron drip line

Run every cell **from the top, in order**. Each part uses results from the parts above it.

Plain-language explanations of every step are in the `docs/` folder of the repository.

## Phase 0 and Phase 1: build the dataset

Download the AME2020 mass table, keep only real measurements, and compute the binding energy of every nucleus.

In [ ]:
url <- "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt"
if (!file.exists("mass_1.mas20.txt")) download.file(url, "mass_1.mas20.txt")
lines <- readLines("mass_1.mas20.txt")
length(lines)

In [ ]:
cat(head(lines, 40), sep = "\n")

1    a0dsskgw                                 A T O M I C   M A S S   A D J U S T M E N T
0                                                     DATE  3 Mar 2021 TIME 22:41
0        *********************                               A=   0 TO 295
         * file : mass.mas20 *
         *********************

   This is one file out of a series of 3 files published in:
       "The Ame2020 atomic mass evaluation (I)"   by W.J.Huang, M.Wang, F.G.Kondev, G.Audi and S.Naimi
           Chinese Physics C45, 030002, March 2021.
       "The Ame2020 atomic mass evaluation (II)"  by M.Wang, W.J.Huang, F.G.Kondev, G.Audi and S.Naimi
           Chinese Physics C45, 030003, March 2021.
                       for files : mass.mas20 : atomic masses
                                   rct1.mas20 : react and sep energies,  part 1
                                   rct2.mas20 : react and sep energies,  part 2
   A fourth file  is the "Rounded" version of the atomic mass table (the first file)
            

In [ ]:
lines <- readLines("mass_1.mas20.txt")

# Data rows: N, Z, A must all parse as integers in cols 5-19
is_data <- nchar(lines) > 100 &
           grepl("^\\s*\\d+\\s+\\d+\\s+\\d+\\s*$", substr(lines, 5, 19))

rows <- lines[is_data]
cat("Header lines:", sum(!is_data), "\n")   # expect 36
cat("Data rows:   ", length(rows), "\n")    # expect 3558

# Estimated values: '#' in the mass excess field or its uncertainty
est <- grepl("#", substr(rows, 29, 54), fixed = TRUE)
cat("Estimated (excluded):", sum(est), "\n")
cat("Measured (kept):     ", sum(!est), "\n")

m <- rows[!est]

num <- function(x, a, b) as.numeric(substr(x, a, b))

df <- data.frame(
  N        = num(m,   5,   9),
  Z        = num(m,  10,  14),
  A        = num(m,  15,  19),
  D        = num(m,  29,  42),   # mass excess, keV
  sigma    = num(m,  43,  54),   # its uncertainty, keV
  BA_table = num(m,  55,  67)    # AME's own B/A, for checking
)

DH <- 7288.97106
Dn <- 8071.31806

df$B  <- df$Z * DH + df$N * Dn - df$D
df$BA <- df$B / df$A

Header lines: 36 
Data rows:    3558 
Estimated (excluded): 1008 
Measured (kept):      2550 


In [ ]:
d <- df$BA - df$BA_table
summary(d)
sum(abs(d) > 1, na.rm = TRUE)   # count exceeding your 1 keV tolerance

      Min.    1st Qu.     Median       Mean    3rd Qu.       Max. 
-1.067e-04 -2.854e-05 -3.127e-06 -3.694e-06  2.000e-05  6.476e-05 

[1] 0

In [ ]:
subset(df, Z == 26 & A == 56)$BA   # ~8790.34
subset(df, Z == 28 & A == 62)$BA   # ~8794.53

[1] 8790.356

[1] 8794.556

### The binding energy curve

In [ ]:
plot(df$A, df$BA, pch = 20, cex = 0.4,
     xlab = "Mass number A",
     ylab = "Binding energy per nucleon (keV)",
     main = "AME2020, 2550 measured nuclides")

peak <- df[which.max(df$BA), ]
points(peak$A, peak$BA, col = "red", pch = 1, cex = 2)
cat("Maximum: Z =", peak$Z, " A =", peak$A, " B/A =", peak$BA, "\n")

head(df[order(-df$BA), c("Z", "N", "A", "BA")], 5)

In [ ]:
key <- paste(df$Z, df$A)
df$Sn  <- df$B - df$B[match(paste(df$Z, df$A - 1), key)]
df$S2n <- df$B - df$B[match(paste(df$Z, df$A - 2), key)]

cat("Sn computable: ", sum(!is.na(df$Sn)),  "of", nrow(df), "\n")
cat("S2n computable:", sum(!is.na(df$S2n)), "of", nrow(df), "\n")
cat("Negative S2n:  ", sum(df$S2n < 0, na.rm = TRUE), "\n")

Sn computable:  2393 of 2550 
S2n computable: 2301 of 2550 
Negative S2n:   9 


### Phase 1, step 6: cross-check against the AME's own separation energies

The file `rct1.mas20.txt` lists the two-neutron separation energy **S2n** that the AME team computed themselves.
If our numbers match theirs, our binding energies, our neighbour lookups and our subtraction are all correct.

Layout of a data row (from the file header, format `a1,i3,1x,a3,i3,1x,6(f12.4,f10.4)`):

| characters | contents |
|---|---|
| 1 | control character |
| 2–4 | A |
| 6–8 | element symbol |
| 9–11 | Z |
| 13–24 | S2n (keV) |
| 25–34 | its uncertainty (keV) |

A `#` in place of the decimal point means an estimate; a `*` means it could not be calculated.

In [ ]:
url1 <- "https://www-nds.iaea.org/amdc/ame2020/rct1.mas20.txt"
if (!file.exists("rct1.mas20.txt")) download.file(url1, "rct1.mas20.txt")
r1 <- readLines("rct1.mas20.txt")
length(r1)

cat(head(r1, 40), sep = "\n")

In [ ]:
# Keep only data rows: A and Z must both end in a digit
rows1 <- r1[grepl("^.[ 0-9]{2}[0-9] [A-Za-z ]{3}[ 0-9]{2}[0-9] ", r1)]
cat("Data rows in rct1:", length(rows1), "\n")   # should equal the 3558 in the mass table

rct <- data.frame(
  A        = as.integer(substr(rows1,  2,  4)),
  Z        = as.integer(substr(rows1,  9, 11)),
  S2n_text = substr(rows1, 13, 24)
)

# Classify each S2n entry: measured, estimated (#), or not calculable (*)
rct$kind <- ifelse(grepl("*", rct$S2n_text, fixed = TRUE), "not calculable",
            ifelse(grepl("#", rct$S2n_text, fixed = TRUE), "estimated", "measured"))
table(rct$kind)

In [ ]:
# Line up the AME's S2n with ours, nucleus by nucleus
rct$S2n_AME <- ifelse(rct$kind == "measured", suppressWarnings(as.numeric(rct$S2n_text)), NA)
df$S2n_AME  <- rct$S2n_AME[match(paste(df$Z, df$A), paste(rct$Z, rct$A))]

# Does every nucleus have an S2n in exactly the same cases as the AME?
table(ours = !is.na(df$S2n), AME = !is.na(df$S2n_AME))

# How far apart are the numbers where both exist?
diff_S2n <- df$S2n - df$S2n_AME
cat("Compared:", sum(!is.na(diff_S2n)), "nuclei\n")
cat("Largest difference:", max(abs(diff_S2n), na.rm = TRUE), "keV\n")

### The nuclei with negative S2n

A negative S2n means the nucleus **cannot hold its last two neutrons**. These nuclei are past the drip line.
They have still been measured, as short-lived resonances, and every one of them is light.

In [ ]:
unbound <- df[!is.na(df$S2n) & df$S2n < 0, c("Z", "N", "A", "S2n")]
unbound[order(unbound$Z), ]

## Phase 2: fit the five-term formula

$$B(A,Z) = a_V A - a_S A^{2/3} - a_C \frac{Z(Z-1)}{A^{1/3}} - a_A \frac{(A-2Z)^2}{A} + \delta$$

Every constant multiplies something we can compute from A and Z alone, so the fit is a straight
linear least-squares problem. We build the five columns, then solve.

**Choices made here (see `docs/DECISIONS.md`):**
- The main fit uses nuclei with **A ≥ 20**. The liquid-drop picture barely applies to smaller nuclei.
- Everything stays in **keV**. The constants are converted to MeV only when printed.
- There is **no intercept**. The formula has no constant term.

In [ ]:
# The five design columns. Row = nucleus, column = what each constant multiplies.
design <- function(Z, A) {
  parity <- ifelse(A %% 2 == 1, 0,            # odd A: no pairing term
            ifelse(Z %% 2 == 0, +1, -1))      # even-even: +1, odd-odd: -1
  cbind(aV = A,
        aS = -A^(2/3),
        aC = -Z * (Z - 1) / A^(1/3),          # Z(Z-1), not Z^2: a proton does not repel itself
        aA = -(A - 2 * Z)^2 / A,
        aP = parity / sqrt(A))
}

# Quick check on iron-56 (Z = 26, even-even)
design(26, 56)

In [ ]:
# Weighted least squares, written out as matrix steps.
#   w = one weight per nucleus (all 1 for uniform weighting)
# Returns the constants, the raw (X'WX)^-1, and chi-squared per degree of freedom.
fit_semf <- function(d, w) {
  X   <- design(d$Z, d$A)
  y   <- d$B
  XtW <- t(X * w)                        # X' W   (W is diagonal, so multiply each row by its weight)
  C0  <- solve(XtW %*% X)                # (X' W X)^-1
  p   <- drop(C0 %*% XtW %*% y)          # the fitted constants, keV
  r   <- y - drop(X %*% p)               # residuals: measured minus predicted
  dof <- nrow(X) - ncol(X)
  list(p = p, C0 = C0, r = r, n = nrow(X),
       chi2dof = sum(w * r^2) / dof,
       rms = sqrt(mean(r^2)))
}

fitset <- subset(df, A >= 20)
cat("Nuclei in the fit:", nrow(fitset), " (left out with A < 20:", sum(df$A < 20), ")\n")

In [ ]:
# Fit 1: uniform weighting
uni <- fit_semf(fitset, rep(1, nrow(fitset)))

# With equal weights, the size of the scatter is not known in advance, so it is estimated
# from the residuals themselves: C = s^2 (X'X)^-1, where s^2 = residual variance.
C_uni <- uni$C0 * uni$chi2dof

cat("Root-mean-square residual:", round(uni$rms), "keV\n\n")
print(round(rbind(value_MeV = uni$p / 1000,
                  std_error_MeV = sqrt(diag(C_uni)) / 1000), 4))

# Same answer from R's built-in regression? (should print a number near zero)
X <- design(fitset$Z, fitset$A)
max(abs(coef(lm(fitset$B ~ 0 + X)) - uni$p))

### The covariance matrix, as correlations

1 means two constants move together completely, and 0 means not at all.

In [ ]:
round(cov2cor(C_uni), 3)

### Compare with published values

Typical published ranges, in MeV (from the project manual):

In [ ]:
published <- data.frame(
  constant = c("aV", "aS", "aC", "aA", "aP"),
  low  = c(15.5, 16.8, 0.70, 23.0, 11),
  high = c(15.8, 18.3, 0.72, 23.7, 12)
)
published$ours    <- round(uni$p / 1000, 3)
published$inside  <- published$ours >= published$low & published$ours <= published$high
published

### Fit 2: inverse-variance weighting with a floor

Each nucleus gets weight $1/\sigma^2$, but no σ may be smaller than a chosen **floor**.
There is no single right floor, so we try four and see how much the answer moves.

Two things to watch:
1. **χ²/dof** (chi-squared per degree of freedom). It would be about 1 if the formula missed only by the measurement errors. It is far bigger, because the formula itself is wrong by a few MeV.
2. **Raw vs scaled standard errors.** The raw $(X^TWX)^{-1}$ trusts the measurement errors. The scaled version multiplies by χ²/dof, which accounts for how badly the formula really fits.

In [ ]:
floors <- c(1, 10, 100, 1000)   # keV
sens <- do.call(rbind, lapply(floors, function(fl) {
  s <- pmax(fitset$sigma, fl)
  f <- fit_semf(fitset, 1 / s^2)
  data.frame(weighting = "floored", floor_keV = fl,
             chi2_per_dof = f$chi2dof, rms_keV = f$rms,
             aV = f$p[1] / 1000, aS = f$p[2] / 1000, aC = f$p[3] / 1000,
             aA = f$p[4] / 1000, aP = f$p[5] / 1000,
             raw_se_aS    = sqrt(f$C0[2, 2]) / 1000,
             scaled_se_aS = sqrt(f$C0[2, 2] * f$chi2dof) / 1000)
}))
uniform_row <- data.frame(weighting = "uniform", floor_keV = NA,
                          chi2_per_dof = NA, rms_keV = uni$rms,
                          aV = uni$p[1] / 1000, aS = uni$p[2] / 1000, aC = uni$p[3] / 1000,
                          aA = uni$p[4] / 1000, aP = uni$p[5] / 1000,
                          raw_se_aS = NA, scaled_se_aS = sqrt(C_uni[2, 2]) / 1000)
sens <- rbind(uniform_row, sens)
rownames(sens) <- NULL
format(sens, digits = 4)

In [ ]:
# How far does the choice of weighting move each constant, compared with its own error bar?
spread <- apply(sens[, c("aV", "aS", "aC", "aA", "aP")], 2, function(x) max(x) - min(x))
se     <- sqrt(diag(C_uni)) / 1000
round(rbind(spread_MeV = spread, std_error_MeV = se, spread_in_std_errors = spread / se), 3)

### Fit 3: does the A ≥ 20 cutoff matter?

In [ ]:
allset  <- subset(df, Z >= 1 & A - Z >= 1)    # every nucleus except the free neutron and hydrogen-1
all_uni <- fit_semf(allset, rep(1, nrow(allset)))
print(round(rbind("A >= 20"    = uni$p / 1000,
                  "all nuclei" = all_uni$p / 1000,
                  "shift in std errors" = (all_uni$p - uni$p) / sqrt(diag(C_uni))), 3))
cat("Nuclei:", uni$n, "vs", all_uni$n, "\n")
cat("rms: A >= 20:", round(uni$rms), "keV;  all nuclei:", round(all_uni$rms), "keV\n")

### A first look at what the formula misses

Phase 3 studies these residuals properly. This plot is a preview: if the formula captured everything,
the points would be a flat, patternless band around zero. The red dotted lines mark the magic numbers 28, 50, 82 and 126.

In [ ]:
plot(fitset$N, uni$r / 1000, pch = 20, cex = 0.4,
     xlab = "Neutron number N", ylab = "Measured minus predicted B (MeV)",
     main = "Five-term formula residuals (uniform weighting, A >= 20)")
abline(h = 0, col = "grey")
abline(v = c(28, 50, 82, 126), col = "red", lty = 3)

## Phase 3: what the formula misses, and a shell correction

Three questions:
1. Do the magic numbers show up in the residuals **by themselves**, without being told where to look?
2. Does a sixth term for shell structure fix them?
3. **Does it help or hurt far from stability**, where the drip line is?

The choices for this phase (D9–D11 in `docs/DECISIONS.md`) were written down before the six-term fit was first run.

In [ ]:
# Residuals from the uniform five-term fit (Phase 2), in MeV
fitset$r5 <- uni$r / 1000

par(mfrow = c(1, 2))
plot(fitset$Z, fitset$r5, pch = 20, cex = 0.3, xlab = "Proton number Z",
     ylab = "Measured minus predicted B (MeV)", main = "Residuals by Z")
abline(h = 0, col = "grey"); abline(v = c(20, 28, 50, 82), col = "red", lty = 3)
plot(fitset$N, fitset$r5, pch = 20, cex = 0.3, xlab = "Neutron number N",
     ylab = "", main = "Residuals by N")
abline(h = 0, col = "grey"); abline(v = c(20, 28, 50, 82, 126), col = "red", lty = 3)
par(mfrow = c(1, 1))

### Finding the magic numbers from the data alone

For each neutron number N, average the residuals of all nuclei with that N. Do the same for each Z.
A **peak** is a value higher than every other value within a window either side of it.
We try windows of ±4, ±6 and ±8 to see which peaks are robust.

In [ ]:
mean_by_N <- tapply(fitset$r5, fitset$N, mean)
mean_by_Z <- tapply(fitset$r5, fitset$Z, mean)

peaks <- function(m, window) {
  v <- as.numeric(names(m))
  is_peak <- sapply(seq_along(m), function(i) all(m[i] >= m[abs(v - v[i]) <= window]))
  paste(v[is_peak], collapse = ", ")
}

data.frame(window = c(4, 6, 8),
           peaks_in_N = sapply(c(4, 6, 8), function(w) peaks(mean_by_N, w)),
           peaks_in_Z = sapply(c(4, 6, 8), function(w) peaks(mean_by_Z, w)))

In [ ]:
par(mfrow = c(1, 2))
plot(as.numeric(names(mean_by_N)), mean_by_N, type = "h", xlab = "N",
     ylab = "Average residual (MeV)", main = "Average residual at each N")
abline(v = c(20, 28, 50, 82, 126), col = "red", lty = 3)
plot(as.numeric(names(mean_by_Z)), mean_by_Z, type = "h", xlab = "Z",
     ylab = "", main = "Average residual at each Z")
abline(v = c(20, 28, 50, 82), col = "red", lty = 3)
par(mfrow = c(1, 1))

### The shell term (decision D9)

$$E_\text{shell} = a_\text{sh}\,[f(Z) + f(N)], \qquad f(n) = \frac{\nu\,(D-\nu)}{D}$$

- ν = how far n is past the magic number below it
- D = the size of that shell, from one magic number to the next

So f is **0 at every magic number** and **largest halfway through a shell**.

In [ ]:
magic_N <- c(0, 2, 8, 20, 28, 50, 82, 126, 184)   # 184: predicted next neutron shell
magic_Z <- c(0, 2, 8, 20, 28, 50, 82, 126)        # 126: predicted next proton shell
# (0 is only a starting point for the smallest numbers; every nucleus in the fit has Z, N >= 5)

shell_f <- function(n, magic) {
  i     <- findInterval(n, magic)      # which shell n is in
  lower <- magic[i]
  D     <- magic[i + 1] - lower
  nu    <- n - lower
  nu * (D - nu) / D
}

# Checks: 0 at magic numbers, 8 halfway between 50 and 82 (16 * 16 / 32)
shell_f(c(28, 50, 66, 82, 126), magic_N)

n <- 0:184
plot(n, shell_f(n, magic_N), type = "l", xlab = "n", ylab = "f(n)",
     main = "Shell term shape: zero at magic numbers, peak mid-shell")
abline(v = magic_N, col = "red", lty = 3)

In [ ]:
# A general weighted least-squares fit for any design matrix (same steps as fit_semf)
fit_lin <- function(X, y, w = rep(1, length(y))) {
  XtW <- t(X * w)
  C0  <- solve(XtW %*% X)
  p   <- drop(C0 %*% XtW %*% y)
  r   <- y - drop(X %*% p)
  chi2dof <- sum(w * r^2) / (nrow(X) - ncol(X))
  list(p = p, C = C0 * chi2dof, r = r, rms = sqrt(mean(r^2)))
}

design6 <- function(Z, A) {
  cbind(design(Z, A), ash = shell_f(Z, magic_Z) + shell_f(A - Z, magic_N))
}

five <- fit_lin(design(fitset$Z, fitset$A),  fitset$B)
six  <- fit_lin(design6(fitset$Z, fitset$A), fitset$B)

max(abs(five$p - uni$p))   # same as Phase 2? (should be ~0)

cat("rms residual, five terms:", round(five$rms), "keV\n")
cat("rms residual, six terms: ", round(six$rms),  "keV\n")
cat("improvement:", round(100 * (1 - six$rms / five$rms), 1), "%\n\n")

round(rbind(five_terms_MeV = c(five$p, ash = NA) / 1000,
            six_terms_MeV  = six$p / 1000,
            six_std_error  = sqrt(diag(six$C)) / 1000), 4)

In [ ]:
# How is the shell constant tied to the others?
round(cov2cor(six$C), 3)

### Which magic numbers does the correction capture?

Average residual of the nuclei sitting **exactly at** each magic number, before and after.
Closer to zero is better.

In [ ]:
fitset$r6 <- six$r / 1000
at_magic <- do.call(rbind, lapply(c(20, 28, 50, 82, 126), function(m) {
  rbind(data.frame(count = "N", magic = m, nuclei = sum(fitset$N == m),
                   before_MeV = mean(fitset$r5[fitset$N == m]),
                   after_MeV  = mean(fitset$r6[fitset$N == m])),
        data.frame(count = "Z", magic = m, nuclei = sum(fitset$Z == m),
                   before_MeV = if (any(fitset$Z == m)) mean(fitset$r5[fitset$Z == m]) else NA,
                   after_MeV  = if (any(fitset$Z == m)) mean(fitset$r6[fitset$Z == m]) else NA))
}))
at_magic <- at_magic[at_magic$nuclei > 0, ]
rownames(at_magic) <- NULL
format(at_magic, digits = 3)

In [ ]:
par(mfrow = c(1, 2))
lim <- range(c(fitset$r5, fitset$r6))
plot(fitset$N, fitset$r5, pch = 20, cex = 0.3, ylim = lim, xlab = "N",
     ylab = "Measured minus predicted B (MeV)", main = "Five terms")
abline(h = 0, col = "grey"); abline(v = c(28, 50, 82, 126), col = "red", lty = 3)
plot(fitset$N, fitset$r6, pch = 20, cex = 0.3, ylim = lim, xlab = "N",
     ylab = "", main = "Six terms (with shell correction)")
abline(h = 0, col = "grey"); abline(v = c(28, 50, 82, 126), col = "red", lty = 3)
par(mfrow = c(1, 1))

### The far-from-stability test (decision D11)

For every element with at least 8 measured isotopes, **hide its most neutron-rich isotopes**.
Fit both formulas to everything else, then predict the hidden ones.
This asks: when we step outward toward the drip line, does the correction help or hurt?

In [ ]:
edge_test <- function(k) {
  # rank isotopes within each element: 1 = most neutron-rich
  rank_from_edge <- ave(-fitset$N, fitset$Z, FUN = function(x) rank(x, ties.method = "first"))
  isotopes       <- ave(fitset$N, fitset$Z, FUN = length)
  hidden <- isotopes >= 8 & rank_from_edge <= k
  train <- fitset[!hidden, ]; test <- fitset[hidden, ]

  f5 <- fit_lin(design(train$Z, train$A),  train$B)
  f6 <- fit_lin(design6(train$Z, train$A), train$B)
  miss5 <- (test$B - drop(design(test$Z, test$A)  %*% f5$p)) / 1000
  miss6 <- (test$B - drop(design6(test$Z, test$A) %*% f6$p)) / 1000
  rms <- function(x) sqrt(mean(x^2))

  by_rank <- data.frame(hidden = k, step_from_edge = rank_from_edge[hidden], miss5, miss6)
  out <- aggregate(cbind(miss5, miss6) ~ step_from_edge, by_rank, rms)
  out <- rbind(data.frame(step_from_edge = "all", miss5 = rms(miss5), miss6 = rms(miss6)), out)
  data.frame(hidden = k, nuclei_hidden = nrow(test), step_from_edge = out$step_from_edge,
             rms_five_terms_MeV = round(out$miss5, 2), rms_six_terms_MeV = round(out$miss6, 2),
             six_better = out$miss6 < out$miss5)
}

cat("In-sample rms for comparison: five terms", round(five$rms / 1000, 2),
    "MeV, six terms", round(six$rms / 1000, 2), "MeV\n\n")
edge_test(4)                       # the main test
rbind(edge_test(2)[1, ], edge_test(6)[1, ])   # sensitivity: hide 2 or 6 (overall rms only)

In [ ]:
# Where does the correction help or hurt on the hidden edge? Light vs heavy elements.
k <- 4
rank_from_edge <- ave(-fitset$N, fitset$Z, FUN = function(x) rank(x, ties.method = "first"))
hidden <- ave(fitset$N, fitset$Z, FUN = length) >= 8 & rank_from_edge <= k
train <- fitset[!hidden, ]; test <- fitset[hidden, ]
f5 <- fit_lin(design(train$Z, train$A),  train$B)
f6 <- fit_lin(design6(train$Z, train$A), train$B)
test$miss5 <- (test$B - drop(design(test$Z, test$A)  %*% f5$p)) / 1000
test$miss6 <- (test$B - drop(design6(test$Z, test$A) %*% f6$p)) / 1000
test$region <- cut(test$Z, c(0, 20, 50, 82, 200), labels = c("Z <= 20", "Z 21-50", "Z 51-82", "Z > 82"))
aggregate(cbind(miss5, miss6) ~ region, test, function(x) round(sqrt(mean(x^2)), 2))

## Phase 4: Garvey–Kelson relations

A completely different approach. Instead of one formula fitted to the whole chart, use
**local** relations: certain combinations of six neighbouring nuclei, added with alternating
signs, come out at almost exactly zero.

Three questions:
1. Which arrangement of the six is the right one? (The manual is explicit that this must be
   checked against the data before anything is built on it.)
2. How accurate are the relations where every mass is measured?
3. **Used to predict outward, how fast does the error grow, and how far out do they keep
   beating the global formula?**

The choices for this phase are D12–D16 in `docs/DECISIONS.md`.

### Finding the relations instead of trusting an index list

All six nuclei fit inside a 3×3 block of the chart. Rather than copy an index arrangement and
hope, take every arrangement that uses each proton row and each neutron column once on each
side — there are six — and see which ones cancel.

In [ ]:
# Any six nuclei sitting in a 3x3 block of the chart: proton offsets 0, 1, 2 and
# neutron offsets 0, 1, 2. A candidate relation takes three of the nine cells with
# a plus sign and three with a minus, using each proton row and each neutron column
# once on each side. Each row below is the neutron offset chosen for proton
# offsets 0, 1 and 2.
perms <- rbind(c(0,1,2), c(0,2,1), c(1,0,2), c(1,2,0), c(2,0,1), c(2,1,0))

# The alternating sum, for every anchor in `at`, reading masses out of `src`.
gk_sum <- function(plus, minus, at, src = df, col = "B") {
  key <- paste(src$Z, src$N)
  val <- src[[col]]
  look <- function(Z, N) val[match(paste(Z, N), key)]
  s <- 0
  for (dz in 0:2)
    s <- s + look(at$Z + dz, at$N + plus[dz + 1]) - look(at$Z + dz, at$N + minus[dz + 1])
  s
}

anchors <- df[, c("Z", "N", "A")]

candidates <- NULL
for (i in 1:5) for (j in (i + 1):6) {
  if (!all(perms[i, ] != perms[j, ])) next      # the two sides may not share a cell
  s <- gk_sum(perms[i, ], perms[j, ], anchors)
  candidates <- rbind(candidates, data.frame(
    plus = paste(perms[i, ], collapse = ""), minus = paste(perms[j, ], collapse = ""),
    hexagons = sum(!is.na(s)),
    rms_keV = round(sqrt(mean(s^2, na.rm = TRUE))),
    median_abs_keV = round(median(abs(s), na.rm = TRUE))))
}
candidates[order(candidates$rms_keV), ]

Two of the six come out at about 100 keV in the median. The other four sit at about 1 MeV,
ten times worse, which is what an arrangement that fails to cancel the smooth part of the mass
looks like. The two that work are the pair Garvey and Kelson published: `120/201` is the
**transverse** relation, `021/102` the **longitudinal** one.

Why those two and not the others: both cancel any mass that can be written as a function of Z
plus a function of N plus a function of A. The transverse does it exactly. The longitudinal
matches the first two moments of A, so a smooth dependence on A survives only at third order.
The remaining four cancel nothing beyond the terms linear in Z and N.

The check below is the one the manual asks for: write the published transverse relation out
term by term and confirm it is the same six cells with the same signs.

In [ ]:
# The published transverse relation, written out term by term, to check the indices:
#   B(Z+2,N-2) - B(Z,N) + B(Z,N-1) - B(Z+1,N-2) + B(Z+1,N) - B(Z+2,N-1)
key <- paste(df$Z, df$N)
Bat <- function(Z, N) df$B[match(paste(Z, N), key)]
published <- Bat(df$Z + 2, df$N - 2) - Bat(df$Z,     df$N)     + Bat(df$Z,     df$N - 1) -
             Bat(df$Z + 1, df$N - 2) + Bat(df$Z + 1, df$N)     - Bat(df$Z + 2, df$N - 1)

# The same six cells, with the block anchored two neutrons lower
mine <- gk_sum(c(1,2,0), c(2,0,1), data.frame(Z = df$Z, N = df$N - 2))
cat("largest disagreement with the published form:", max(abs(published - mine), na.rm = TRUE), "keV\n")
cat("computable in exactly the same places:", identical(is.na(published), is.na(mine)), "\n\n")

# Masses, mass excesses and binding energies differ by terms linear in Z, N and A,
# and both relations cancel those exactly. So all three must give the same answer.
for (p in list(c(4,5), c(2,3))) {
  a <- gk_sum(perms[p[1],], perms[p[2],], anchors, col = "B")
  b <- gk_sum(perms[p[1],], perms[p[2],], anchors, col = "D")
  cat(paste(perms[p[1],], collapse = ""), "/", paste(perms[p[2],], collapse = ""),
      ": binding energy vs mass excess, largest disagreement",
      signif(max(abs(a + b), na.rm = TRUE), 3), "keV\n")
}

### The relations as they were published

The enumeration above says which six cells cancel. It does not by itself prove that
those are the relations Garvey and Kelson wrote down, or that the residual carries
the sign theirs does. Both are checked here against the published equations, which
are written with the arguments in the order M(N, Z) and in terms of masses rather
than binding energies:

> M(N+2, Z−2) − M(N, Z) + M(N, Z−1) − M(N+1, Z−2) + M(N+1, Z) − M(N+2, Z−1) = 0  (transverse)
>
> M(N+2, Z) − M(N, Z−2) + M(N+1, Z−2) − M(N+2, Z−1) + M(N, Z−1) − M(N+1, Z) = 0  (longitudinal)

Sources: Garvey and Kelson, *Physical Review Letters* **16**, 197 (1966); Garvey,
Gerace, Jaffe, Talmi and Kelson, *Reviews of Modern Physics* **41**, S1 (1969).
The pair is quoted in this form, with a derivation of why they work, by
Piekarewicz, Centelles, Roca-Maza and Viñas, *Garvey–Kelson relations for nuclear
charge radii*, Eur. Phys. J. A **46**, 379 (2010), arXiv:0912.0503, Eqs. (1a)
and (1b).

In [ ]:
# Both relations as Garvey and Kelson published them. Note the argument order:
# the literature writes M(N, Z), and the masses are atomic masses, for which the
# mass excess stands in here (they differ by A atomic mass units, a term linear
# in A, which both relations cancel).
Dat <- function(N, Z) df$D[match(paste(Z, N), key)]

# Eq. (1a), the transverse relation
gk1a <- Dat(df$N + 2, df$Z - 2) - Dat(df$N,     df$Z)     +
        Dat(df$N,     df$Z - 1) - Dat(df$N + 1, df$Z - 2) +
        Dat(df$N + 1, df$Z)     - Dat(df$N + 2, df$Z - 1)

# Eq. (1b), the longitudinal relation
gk1b <- Dat(df$N + 2, df$Z)     - Dat(df$N,     df$Z - 2) +
        Dat(df$N + 1, df$Z - 2) - Dat(df$N + 2, df$Z - 1) +
        Dat(df$N,     df$Z - 1) - Dat(df$N + 1, df$Z)

# The same six cells as the two arrangements found above, anchored two protons lower.
low <- data.frame(Z = df$Z - 2, N = df$N)
mine_T <- gk_sum(c(1,2,0), c(2,0,1), low)
mine_L <- gk_sum(c(0,2,1), c(1,0,2), low)

for (p in list(list("transverse  (1a)", gk1a, mine_T), list("longitudinal (1b)", gk1b, mine_L))) {
  cat(sprintf("%-18s same six nuclei: %s   same sign: %s   largest gap: %.2g keV\n",
      p[[1]], identical(is.na(p[[2]]), is.na(p[[3]])),
      max(abs(p[[2]] - p[[3]]), na.rm = TRUE) < 1e-6,
      min(max(abs(p[[2]] - p[[3]]), na.rm = TRUE), max(abs(p[[2]] + p[[3]]), na.rm = TRUE))))
}
cat("\nSo the arrangements found by enumeration are Eqs. (1a) and (1b) themselves.\n")
cat("The sides are labelled the other way round here, and binding energy carries the\n")
cat("opposite sign to a mass, so the two flips cancel and the residuals come out\n")
cat("with the published sign.\n")

Both published equations pick out the same six nuclei as the two arrangements found
by enumeration, and with the same sign, so the residuals reported below are the
published quantity and not its negative.

Two labelling flips happen to cancel on the way there. The enumeration puts the
sides the other way round from Eqs. (1a) and (1b), and binding energy carries the
opposite sign to a mass. Worth stating explicitly, because a single unnoticed flip
would leave every mean residual in this phase pointing the wrong way while every
rms stayed right.

In [ ]:
# Why these two and not the other four: feed each arrangement the smooth five-term
# formula instead of the real masses. Whatever survives is smooth curvature the
# arrangement failed to cancel.
smooth <- df; smooth$B <- drop(design(df$Z, df$A) %*% uni$p)
cat("Each arrangement applied to the smooth formula alone (rms, keV):\n")
for (i in 1:5) for (j in (i + 1):6) {
  if (!all(perms[i, ] != perms[j, ])) next
  cat(sprintf("  %s / %s : %7.0f\n", paste(perms[i, ], collapse = ""),
              paste(perms[j, ], collapse = ""),
              sqrt(mean(gk_sum(perms[i, ], perms[j, ], anchors, src = smooth)^2, na.rm = TRUE))))
}
# The shell term of Phase 3 is a function of Z plus a function of N, so both
# relations should cancel it as well.
smooth6 <- df; smooth6$B <- drop(design6(df$Z, df$A) %*% six$p)
cat("\nWith the shell term added to the smooth formula:\n")
for (p in list(c(4, 5), c(2, 3))) {
  cat(sprintf("  %s / %s : %7.0f\n", paste(perms[p[1], ], collapse = ""),
              paste(perms[p[2], ], collapse = ""),
              sqrt(mean(gk_sum(perms[p[1], ], perms[p[2], ], anchors, src = smooth6)^2, na.rm = TRUE))))
}

In [ ]:
# The transverse relation is not merely approximately a third difference of the
# mass surface. It is exactly one: d3/dZ dN^2 minus d3/dZ^2 dN. Anything whose
# third derivatives vanish - anything linear or quadratic in Z and N - drops out
# of it completely.
at  <- function(dz, dn) df$D[match(paste(df$Z + dz, df$N + dn), key)]
ZN2 <- (at(1,2) - 2*at(1,1) + at(1,0)) - (at(0,2) - 2*at(0,1) + at(0,0))
Z2N <- (at(2,1) - 2*at(1,1) + at(0,1)) - (at(2,0) - 2*at(1,0) + at(0,0))

cat("transverse minus (d3/dZdN^2 - d3/dZ^2dN):",
    signif(max(abs(gk_sum(c(1,2,0), c(2,0,1), anchors, col = "D") - (ZN2 - Z2N)),
               na.rm = TRUE), 2), "keV\n")
cat("The longitudinal relation is not exactly a third difference; it equals the sum\n")
cat("of the same two, plus a fourth-order remainder, which is why it leaves more of\n")
cat("the smooth formula behind (218 keV against 132).\n")

### How accurate are they?

Every block where all six masses are measured, as a distribution.

In [ ]:
transverse   <- list(plus = c(1,2,0), minus = c(2,0,1))
longitudinal <- list(plus = c(0,2,1), minus = c(1,0,2))

hex <- df[, c("Z", "N", "A")]
hex$T <- gk_sum(transverse$plus,   transverse$minus,   hex)
hex$L <- gk_sum(longitudinal$plus, longitudinal$minus, hex)

describe <- function(x, nm) {
  x <- x[!is.na(x)]
  cat(sprintf("%-19s n=%4d  rms=%6.0f  mean=%6.1f  median|.|=%5.0f  90%%|.|=%5.0f  worst=%6.0f\n",
              nm, length(x), sqrt(mean(x^2)), mean(x), median(abs(x)),
              quantile(abs(x), 0.9), max(abs(x))))
}
cat("Alternating sum, in keV:\n")
describe(hex$T, "transverse");   describe(hex$L, "longitudinal")
describe(hex$T[hex$A >= 20], "transverse, A>=20");   describe(hex$L[hex$A >= 20], "longitudinal, A>=20")

cat("\nFor comparison, the global formula's own misses on the fitted nuclei:\n")
cat("  five terms:", round(five$rms), "keV     six terms:", round(six$rms), "keV\n")

par(mfrow = c(1, 2))
for (v in c("T", "L")) {
  hist(pmax(pmin(hex[[v]], 1500), -1500), breaks = 60, col = "grey85", border = NA,
       xlab = "Alternating sum (keV)", main = c(T = "Transverse", L = "Longitudinal")[v])
  abline(v = 0, col = "red")
}
par(mfrow = c(1, 1))

The median miss is about 110 keV, against 2,560 keV for the six-term formula — better by a
factor of twenty. But the rms is 500–600 keV, five times the median, so the distribution has
heavy tails: a handful of light nuclei miss by several MeV. Below A = 20 the relations are
hardly better than the global formula.

The accuracy also depends strongly on where you are. It has to be read inside a mass band,
because the relations are much better in heavy nuclei, and blocks far inside an element's
measured range are heavy ones.

In [ ]:
rms <- function(x) round(sqrt(mean(x^2, na.rm = TRUE)))
both <- subset(hex, !is.na(T) & !is.na(L))

# The relations are far more accurate in heavy nuclei than light ones, so any
# comparison by distance has to be made inside a mass band or it just re-reads that.
both$region <- cut(both$Z, c(0, 20, 50, 82, 200),
                   labels = c("Z<=20", "Z 21-50", "Z 51-82", "Z>82"))
edge_N <- tapply(df$N, df$Z, max)      # most neutron-rich measured isotope of each element
both$from_edge <- edge_N[as.character(both$Z)] - both$N
both$steps <- cut(both$from_edge, c(-1, 5, 10, 20, 200),
                  labels = c("0-5", "6-10", "11-20", ">20"))

cat("rms of the alternating sum in keV, by region and by how far the block sits\n")
cat("inside that element's measured range:\n\n")
tab <- aggregate(cbind(T, L) ~ region + steps, both, rms)
tab$hexagons <- aggregate(T ~ region + steps, both, length)$T
print(tab[order(tab$region, tab$steps), ], row.names = FALSE)

### Using them to predict (decisions D12–D16)

If five of the six masses are known, the relation gives the sixth. Feed each prediction back
in and the process runs outward step by step, which is how these relations are used to reach
nuclei nobody has measured.

The test is the same hold-out as Phase 3 (D11): hide the most neutron-rich isotopes of each
element, chain outward to predict them, and compare against the five- and six-term formulas
refitted without them. Every nucleus counted here is one whose mass is known, so the
prediction can be checked.

Twelve routes reach any one nucleus: two relations, and six choices of which of the six cells
is the unknown one.

In [ ]:
# For one relation, and for each of its six cells taken in turn as the unknown one,
# the offsets and signs of the other five, counted from the unknown cell.
templates <- function(rel) {
  cells <- rbind(cbind(0:2, rel$plus, +1), cbind(0:2, rel$minus, -1))
  lapply(1:6, function(k) {
    o <- cells[-k, , drop = FALSE]
    # the six terms sum to zero, so the unknown is minus the other five over its own sign
    data.frame(dz = o[, 1] - cells[k, 1], dn = o[, 2] - cells[k, 2],
               s = -o[, 3] / cells[k, 3])
  })
}
routes <- unlist(list(T = templates(transverse), L = templates(longitudinal)),
                 recursive = FALSE)

length(routes)          # twelve ways to reach one unknown nucleus
routes[["T1"]]          # one of them, as offsets from the nucleus being predicted

In [ ]:
# Chain outward: predict every nucleus that has five of its six masses in hand,
# add those predictions to the pile, and go round again. `sigma` is each relation's
# own scatter; predictions inherit it, so uncertainty grows with every round.
chain <- function(known, target, sigma) {
  B  <- setNames(known$B, paste(known$Z, known$N))
  V  <- setNames(rep(0, nrow(known)), names(B))    # measurement errors (~6 keV) ignored
  tg <- target[, c("Z", "N", "A", "B")]
  tg$pred <- NA_real_; tg$var <- NA_real_; tg$round <- NA_integer_
  tg$n_routes <- NA_integer_; tg$spread <- NA_real_

  for (g in 1:40) {
    todo <- which(is.na(tg$pred))
    if (!length(todo)) break
    est <- var <- matrix(NA_real_, length(todo), length(routes))
    for (t in seq_along(routes)) {
      o <- routes[[t]]
      acc <- rep(0, length(todo))
      v   <- rep(sigma[[substr(names(routes)[t], 1, 1)]]^2, length(todo))
      for (i in 1:5) {
        k   <- paste(tg$Z[todo] + o$dz[i], tg$N[todo] + o$dn[i])
        acc <- acc + o$s[i] * B[k]
        v   <- v + V[k]
      }
      est[, t] <- acc; var[, t] <- v
    }
    n <- rowSums(!is.na(est))
    if (!any(n > 0)) break                         # nothing else can be reached
    got <- todo[n > 0]; e <- est[n > 0, , drop = FALSE]; v <- var[n > 0, , drop = FALSE]
    tg$pred[got]   <- rowMeans(e, na.rm = TRUE)    # average over every route that works
    tg$var[got]    <- rowMeans(v, na.rm = TRUE)    # routes overlap, so no 1/n reduction
    tg$spread[got] <- apply(e, 1, sd, na.rm = TRUE)
    tg$n_routes[got] <- n[n > 0]
    tg$round[got]  <- g
    B[paste(tg$Z[got], tg$N[got])] <- tg$pred[got]
    V[paste(tg$Z[got], tg$N[got])] <- tg$var[got]
  }
  tg$err   <- (tg$pred - tg$B) / 1000              # MeV
  tg$sigma <- sqrt(tg$var) / 1000
  tg
}

# Hide the k most neutron-rich isotopes of every element that has at least
# `min_iso` of them, exactly as in decision D11, and predict them three ways:
# by chaining the relations, and from the five- and six-term formulas refitted
# without them.
holdout <- function(k, min_iso) {
  from_edge <- ave(-fitset$N, fitset$Z, FUN = function(x) rank(x, ties.method = "first"))
  hidden <- ave(fitset$N, fitset$Z, FUN = length) >= min_iso & from_edge <= k
  train <- fitset[!hidden, ]; test <- fitset[hidden, ]

  # each relation's scatter, from blocks lying wholly inside the training set
  sigma <- list(T = sqrt(mean(gk_sum(transverse$plus,   transverse$minus,   train, train)^2, na.rm = TRUE)),
                L = sqrt(mean(gk_sum(longitudinal$plus, longitudinal$minus, train, train)^2, na.rm = TRUE)))

  f5 <- fit_lin(design(train$Z, train$A),  train$B)
  f6 <- fit_lin(design6(train$Z, train$A), train$B)
  out <- chain(train, test, sigma)
  out$hidden    <- k
  out$from_edge <- from_edge[hidden]
  out$miss5 <- (test$B - drop(design(test$Z, test$A)  %*% f5$p)) / 1000
  out$miss6 <- (test$B - drop(design6(test$Z, test$A) %*% f6$p)) / 1000
  attr(out, "sigma") <- sigma
  out
}

main <- holdout(4, 8)
cat("hidden:", nrow(main), " reached by chaining:", sum(!is.na(main$pred)),
    " unreachable:", sum(is.na(main$pred)), "\n")
cat("relation scatter taken from the training set alone: transverse",
    round(attr(main, "sigma")$T), "keV, longitudinal", round(attr(main, "sigma")$L), "keV\n\n")

got <- subset(main, !is.na(pred))
tab <- aggregate(cbind(err, miss5, miss6) ~ from_edge, got, function(x) round(sqrt(mean(x^2)), 2))
tab$nuclei <- aggregate(err ~ from_edge, got, length)$err
names(tab) <- c("steps_out", "relations", "five_terms", "six_terms", "nuclei")
print(tab[, c("steps_out", "nuclei", "relations", "five_terms", "six_terms")], row.names = FALSE)
cat("\nall together (rms, MeV): relations", round(sqrt(mean(got$err^2)), 2),
    " five terms", round(sqrt(mean(got$miss5^2)), 2),
    " six terms", round(sqrt(mean(got$miss6^2)), 2), "\n")

At the hidden edge the relations are better than the six-term formula by a factor of four to
twelve, and they get better the fewer steps they have to take. 17 of the 364 hidden nuclei
cannot be reached at all: chaining needs five measured neighbours to start from, and the
global formula will always return an answer where the relations return nothing.

### How fast does the error grow?

In [ ]:
# Deeper hold-outs, so the chains have to run further. Pooling them puts many
# nuclei at each chain length. The x axis that matters is the number of chained
# rounds, not the distance from the edge: a nucleus two steps out can still need
# several rounds if the route to it has to go round an unmeasured gap.
deep <- do.call(rbind, lapply(c(4, 8, 12, 16, 20), function(k) {
  r <- subset(holdout(k, k + 4), !is.na(pred))
  r[, c("hidden", "Z", "N", "A", "from_edge", "round", "err", "sigma",
        "n_routes", "spread", "miss5", "miss6")]
}))

growth <- aggregate(cbind(err, miss5, miss6) ~ round, deep, function(x) sqrt(mean(x^2)))
growth$nuclei  <- aggregate(err ~ round, deep, length)$err
growth$med_abs <- aggregate(abs(err) ~ round, deep, median)[, 2]
g <- subset(growth, nuclei >= 30)

print(data.frame(rounds = g$round, nuclei = g$nuclei,
                 median_abs = round(g$med_abs, 2), rms_relations = round(g$err, 2),
                 rms_five = round(g$miss5, 2), rms_six = round(g$miss6, 2)),
      row.names = FALSE)

power <- lm(log(err) ~ log(round), g)
cat("\nrms error against number of rounds, fitted as a power law:\n")
cat("  rms =", round(exp(coef(power)[1]), 3), "MeV x rounds^", round(coef(power)[2], 2), "\n")
cat("  a random walk of independent errors would give an exponent of 0.5\n")

The error grows as **rounds^1.37**, not as the square root. Independent errors accumulating at
random would give an exponent of 0.5. This one is steeper because the errors are not
independent: a prediction is built from its neighbours, and when those neighbours were
themselves predicted, their errors come along and push in the same direction.

The practical consequence is the crossover.

In [ ]:
plot(g$round, g$err, log = "xy", type = "b", pch = 20, ylim = c(0.1, 20),
     xlab = "Chained steps beyond measured nuclei",
     ylab = "rms error in binding energy (MeV)",
     main = "Where the relations stop beating the global formula")
lines(g$round, g$miss6, type = "b", pch = 1, col = "blue")
lines(g$round, g$miss5, type = "b", pch = 1, col = "grey50")
curve(exp(coef(power)[1]) * x^coef(power)[2], add = TRUE, col = "red", lty = 2)
legend("topleft", bty = "n", pch = c(20, 1, 1, NA), lty = c(1, 1, 1, 2),
       col = c("black", "blue", "grey50", "red"),
       legend = c("Garvey-Kelson, chained", "six-term formula", "five-term formula",
                  "power law fit"))

cross <- subset(g, err > miss6)
cat("The six-term formula first overtakes the relations at",
    min(cross$round), "chained steps:\n")
print(data.frame(rounds = cross$round[1:3], nuclei = cross$nuclei[1:3],
                 relations = round(cross$err[1:3], 2), six_terms = round(cross$miss6[1:3], 2)),
      row.names = FALSE)

So the relations win out to about **twelve chained steps** beyond measured territory and lose
after that. This is a real limit on the method, and it matters for Phase 7: for most heavy
elements the drip line is tens of neutrons past the last measured isotope, which is well past
where this advantage runs out.

### Putting an error bar on a chained prediction

Two obvious ways to do it. Both fail, in opposite directions, and how they fail is more useful
than either number.

In [ ]:
# Two ways of putting an error bar on a chained prediction, both of which fail.
#
# 1. Propagate the relation's own scatter through the chain. Each round adds the
#    scatter plus the variance of whichever inputs were themselves predicted.
band <- aggregate(cbind(sigma, abs(err)) ~ round, deep, mean)
band$rms      <- aggregate(err ~ round, deep, function(x) sqrt(mean(x^2)))$err
band$inside_1 <- aggregate(I(abs(deep$err) <= deep$sigma) ~ deep$round, data.frame(deep), mean)[, 2]
band$nuclei   <- aggregate(err ~ round, deep, length)$err
b <- subset(band, nuclei >= 30)
cat("Propagated band (a 1-sigma band should hold about 68 % of them):\n")
print(data.frame(rounds = b$round, nuclei = b$nuclei,
                 mean_sigma = round(b$sigma, 2), rms_error = round(b$rms, 2),
                 inside_1_sigma = round(b$inside_1, 2)), row.names = FALSE)
cat("\nOverall inside 1 sigma:", round(mean(abs(deep$err) <= deep$sigma), 2),
    "- the band is too wide, and grows exponentially while the real error grows as a power.\n")

# 2. Use the disagreement between the routes that reached the same nucleus.
multi <- subset(deep, n_routes >= 2 & !is.na(spread) & spread > 0)
multi$ratio <- abs(multi$err) * 1000 / multi$spread
cat("\nDisagreement between routes, for the", nrow(multi), "nuclei reached more than one way:\n")
sp <- aggregate(ratio ~ round, multi, function(x) round(median(x), 1))
sp$nuclei <- aggregate(ratio ~ round, multi, length)$ratio
sp$inside_2 <- aggregate(I(abs(multi$err) * 1000 <= 2 * multi$spread) ~ multi$round,
                         data.frame(multi), function(x) round(mean(x), 2))[, 2]
names(sp) <- c("rounds", "median_error_over_spread", "nuclei", "inside_2_x_spread")
print(head(subset(sp, nuclei >= 30), 12), row.names = FALSE)
cat("\nOverall inside twice the spread:",
    round(mean(abs(multi$err) * 1000 <= 2 * multi$spread), 2),
    "- far too narrow. Routes that share an already-predicted\nneighbour inherit its error, so they agree with each other while all being wrong.\n")

Neither works. Propagating the relation's own scatter multiplies the variance by roughly five
per round, so the band grows exponentially while the real error grows as a power — by twenty
steps it is 700 MeV wide and completely useless. Route disagreement fails the other way:
routes that pass through the same already-predicted neighbour inherit its error, so they agree
with each other while all being wrong together.

What is left is the measured curve itself. The hold-out gives an honest error bar as a
function of chain length, **rms ≈ 0.13 MeV × rounds^1.37**, and that is what Phase 7 will use.
It is an average over the whole chart, though, and the region-by-region table above shows the
spread around it is large.

### Each relation on its own

In [ ]:
# Each relation on its own, and what having more than one route is worth.
for (nm in c("T", "L")) {
  keep <- routes
  routes <- keep[grep(paste0("^", nm), names(keep))]
  r <- subset(holdout(4, 8), !is.na(pred))
  cat(sprintf("%-12s alone: reached %3d of 364  rms %5.2f MeV  median|error| %5.2f MeV\n",
              c(T = "transverse", L = "longitudinal")[nm], nrow(r),
              sqrt(mean(r$err^2)), median(abs(r$err))))
  routes <- keep
}
cat(sprintf("%-12s  both: reached %3d of 364  rms %5.2f MeV  median|error| %5.2f MeV\n", "",
            nrow(got), sqrt(mean(got$err^2)), median(abs(got$err))))

cat("\nPredictions reached by only one route against those reached by several:\n")
for (i in c(1, 2)) {
  s <- if (i == 1) subset(deep, n_routes == 1) else subset(deep, n_routes >= 2)
  cat(sprintf("  %-16s n=%4d  rms=%5.2f MeV  median|error|=%5.2f MeV\n",
              if (i == 1) "one route" else "two or more", nrow(s),
              sqrt(mean(s$err^2)), median(abs(s$err))))
}

The transverse relation reaches more nuclei (333 of 364 against 259) because its footprint sits
better against a ragged neutron-rich edge. The longitudinal one has the smaller median error.
Together they reach 347, more than either alone, and the rms of the combination is no better
than the transverse alone — averaging the two does not help, it just fills in gaps.

Having more than one route is worth a lot: 0.32 MeV median error against 0.68 MeV for the
nuclei reached only one way. Those are the ones out on a thin filament where nothing can
cross-check them.

## Phase 5: the blind test

Everything up to here has been marked against nuclei that were already in the table.
A fit reports how well it fits; that proves nothing, because a flexible enough
formula always fits. The question is whether it **predicts**.

So: fit only to what was known in 2003, then predict the nuclei measured since.
Seventeen years of experimental work becomes an unseen test set, and most of it is
in the neutron-rich direction, which is where the drip line is.

The rules were fixed in Phase 2 (D8 in `docs/DECISIONS.md`), before any of this was
run: nothing from after the cutoff may touch the fit — not the data, not the
weighting, not the exclusions — every weighting variant gets reported rather than
whichever predicts best, and no choice may be revised after seeing a test result.
D17–D20 add the detail, written down after the two sets were counted and before any
prediction was made.

### Reading AME2003

In [ ]:
url03 <- "https://www-nds.iaea.org/amdc/ame2003/mass.mas03"
if (!file.exists("mass.mas03")) download.file(url03, "mass.mas03")

# mass.mas03 declares its own format:
#   a1,i3,i5,i5,i5,1x,a3,a4,1x,f13.5,f11.5,f11.3,f9.3,...
# N, Z and A are in the same columns as AME2020, but the mass excess is at 29-41
# and its uncertainty at 42-52, two columns narrower than the 2020 file. The
# Phase 1 reader cannot be pointed at this file unchanged.
l03 <- readLines("mass.mas03")
is03 <- nchar(l03) > 100 &
        grepl("^\\s*\\d+\\s+\\d+\\s+\\d+\\s*$", substr(l03, 5, 19))
cat("header lines:", sum(!is03), " data rows:", sum(is03),
    " (the file's own header says 39 and 3179)\n")

r03 <- l03[is03]
est03 <- grepl("#", substr(r03, 29, 52), fixed = TRUE)
cat("estimated in 2003 (excluded):", sum(est03), "   measured in 2003:", sum(!est03), "\n")

old <- data.frame(N = num(r03, 5, 9), Z = num(r03, 10, 14), A = num(r03, 15, 19),
                  D = num(r03, 29, 41), sigma = num(r03, 42, 52),
                  BA_table = num(r03, 53, 63), measured = !est03)

# AME2003's own values, not the 2020 ones: nothing from after the cutoff may enter
# the fit, not even a mass excess constant (D17)
old$B  <- old$Z * 7288.97050 + old$N * 8071.31710 - old$D
old$BA <- old$B / old$A

chk <- subset(old, measured & A > 0)
cat("\nB/A against the 2003 file's own column: largest gap",
    signif(max(abs(chk$BA - chk$BA_table), na.rm = TRUE), 3), "keV over",
    nrow(chk), "nuclei\n")
cat("iron-56 B/A:", round(subset(old, Z == 26 & A == 56)$BA, 2),
    "keV in 2003 against", round(subset(df, Z == 26 & A == 56)$BA, 2), "in 2020\n")

### The two sets

AME2020 had 2,550 measured nuclei after Phase 1 threw out its estimates. The
question is which of them AME2003 already had.

In [ ]:
# `df` holds the 2,550 nuclei measured in AME2020; `old` holds AME2003 with a flag.
k20 <- paste(df$Z, df$N)
k03 <- paste(old$Z, old$N)
known03 <- subset(old, measured)      # everything known in 2003, at every A
df$in03 <- k20 %in% paste(known03$Z, known03$N)

cat("measured in AME2020:             ", nrow(df), "\n")
cat("  also measured in AME2003:      ", sum(df$in03), "\n")
cat("  estimated in AME2003:          ", sum(!df$in03 & k20 %in% k03), "\n")
cat("  absent from AME2003 altogether:", sum(!(k20 %in% k03)), "\n")

# The other direction: measurements AME2020 withdrew
gone <- subset(known03, !(paste(Z, N) %in% k20))
cat("\nmeasured in 2003 but no longer measured in AME2020:", nrow(gone), "\n")
print(gone[order(gone$Z), c("Z", "N", "A", "D", "sigma")], row.names = FALSE, digits = 6)
cat("Most carry 2003 uncertainties of hundreds to thousands of keV. They stay in the\n")
cat("fit set, because someone fitting in 2003 would have used them (D18); a refit\n")
cat("without them is below.\n")

fit03  <- subset(old, measured & A >= 20)          # the A >= 20 cutoff of D5, unchanged
test03 <- subset(df, !in03 & A >= 20)
cat("\nfit set  (AME2003, A >= 20):                  ", nrow(fit03), "nuclei\n")
cat("test set (first measured after 2003, A >= 20):", nrow(test03), "nuclei\n")
cat("  left out of the test set for A < 20:", sum(!df$in03 & df$A < 20), "\n")

### Which direction is the test set in?

In [ ]:
# Seventeen years of progress is not all outward. For each element take the range of
# neutron numbers AME2003 had measured - from everything measured, at every A, or a
# light element with nothing above A = 20 would be miscalled a new element - and sort
# each test nucleus by where it falls relative to that range (D18).
lo03 <- tapply(known03$N, known03$Z, min)
hi03 <- tapply(known03$N, known03$Z, max)
test03$hi <- hi03[as.character(test03$Z)]
test03$lo <- lo03[as.character(test03$Z)]
test03$kind <- ifelse(is.na(test03$hi), "new element",
               ifelse(test03$N > test03$hi, "neutron-rich extension",
               ifelse(test03$N < test03$lo, "proton-rich extension", "gap fill")))
test03$out <- ifelse(test03$kind == "neutron-rich extension", test03$N - test03$hi, NA)

print(table(test03$kind))
cat("\nOnly the neutron-rich extensions test extrapolation toward the drip line.\n")
cat("How far past the 2003 edge they sit:\n")
print(table(steps_past_2003_edge = test03$out))
cat("\nby region:\n")
print(table(region = cut(test03$Z, c(0, 20, 50, 82, 200),
                         labels = c("Z<=20", "Z 21-50", "Z 51-82", "Z>82")),
            kind = test03$kind))

Only 182 of the 326 are neutron-rich extensions. The rest fill gaps inside the 2003
range, sit on the proton-rich side, or belong to twelve superheavy nuclei of elements
AME2003 had nothing measured for at all. All four kinds get reported, but only the
first is a test of extrapolation toward the drip line.

### The blind test

In [ ]:
# Fit to 2003 and predict the test set, with every weighting of D6 reported and
# none of them chosen afterwards (D8).
wts <- list(uniform = rep(1, nrow(fit03)))
for (f in c(1, 10, 100, 1000))
  wts[[paste0("floor_", f, "keV")]] <- 1 / pmax(fit03$sigma, f)^2

X5f <- design(fit03$Z, fit03$A);   X6f <- design6(fit03$Z, fit03$A)
X5t <- design(test03$Z, test03$A); X6t <- design6(test03$Z, test03$A)
rms <- function(x) sqrt(mean(x^2))

blind <- list(); tbl <- NULL
for (nm in names(wts)) for (tag in c("five", "six")) {
  Xf <- if (tag == "five") X5f else X6f
  Xt <- if (tag == "five") X5t else X6t
  f <- fit_lin(Xf, fit03$B, wts[[nm]])
  err <- (test03$B - drop(Xt %*% f$p)) / 1000
  blind[[paste(tag, nm)]] <- list(fit = f, err = err, Xt = Xt)
  tbl <- rbind(tbl, data.frame(formula = tag, weighting = nm,
    fit_rms = round(f$rms / 1000, 2), test_rms = round(rms(err), 2),
    test_median = round(median(abs(err)), 2),
    neutron_rich = round(rms(err[test03$kind == "neutron-rich extension"]), 2),
    gap_fill = round(rms(err[test03$kind == "gap fill"]), 2)))
}
print(tbl[order(tbl$formula, tbl$weighting), ], row.names = FALSE)
cat("\n(every six-term number here is NOT blind - its shell term was shaped on AME2020\n")
cat(" residuals, which include these test nuclei. See D19.)\n")

u5 <- blind[["five uniform"]]
cat("\nThe reference fit (five terms, uniform weighting):\n")
cat("  rms on the nuclei it was fitted to:", round(u5$fit$rms / 1000, 2), "MeV\n")
cat("  rms on nuclei measured after 2003: ", round(rms(u5$err), 2), "MeV   ->",
    round(rms(u5$err) / (u5$fit$rms / 1000), 2), "times worse\n")
cat("  median |error|:", round(median(abs(u5$err)), 2),
    "MeV, so the rms is carried by a tail\n")
cat("  mean error:", round(mean(u5$err), 2),
    "MeV - the new nuclei are more bound than predicted, not less\n")

**The formula is 1.65 times worse on nuclei it has not seen** — 2.98 MeV on the fit
set, 4.90 MeV on the test set. Reporting the fit residual as if it were a prediction
error would have understated the error by that factor.

Two details matter more than the ratio. The median error is only 1.72 MeV, so the rms
is carried by a tail. And the mean error is **+1.29 MeV**, not zero: the nuclei found
since 2003 are systematically *more* bound than the 2003 fit predicted. A formula that
was merely imprecise would miss in both directions equally.

The weighting order is the same as in Phase 2 — uniform and the 1,000 keV floor best,
the 1 keV floor worst — which is at least consistent with D6 having been the right
call for the right reason.

### Did the constants stay inside their error bars?

D7 flagged this as its own known weakness: the residuals are not random, so even the
χ²/dof-scaled error bars are probably too small. Seventeen years of new data is the
direct test.

In [ ]:
# The direct test of D7's known limit: seventeen years of new data arrived. Did the
# constants stay inside the error bars the 2003 fit put on them?
f03 <- fit_lin(X5f, fit03$B)
print(round(rbind(AME2003_MeV   = f03$p / 1000,
                  std_error_MeV = sqrt(diag(f03$C)) / 1000,
                  AME2020_MeV   = uni$p / 1000,
                  shift_in_std_errors = (uni$p - f03$p) / sqrt(diag(f03$C))), 3))
cat("\nFour of the five moved by more than five of their own 2003 standard errors.\n")
cat("The error bars of Phase 2 are scaled by chi-squared per degree of freedom (D7),\n")
cat("which is already a large correction, and they are still far too small.\n")

**Four of the five constants moved by more than five of their own 2003 standard
errors.** The volume term moved 6.4 σ, the surface term 6.9 σ. Nothing about the
2003 fit warned that this would happen; the error bars were already inflated by
χ²/dof, a correction of a factor of hundreds, and they are still far too small to
cover what actually changed.

This is the strongest single result in the project so far. A published constant with
a stated error bar, fitted this way, does not mean what it appears to mean.

### Does the predicted uncertainty on a prediction hold?

In [ ]:
# The band declared in D20: the constants' covariance carried through the design row,
# plus the scatter of the fit itself.
band <- function(k) sqrt(rowSums((k$Xt %*% k$fit$C) * k$Xt) + mean(k$fit$r^2)) / 1000

for (nm in c("five uniform", "six uniform")) {
  k <- blind[[nm]]; s <- band(k); e <- abs(k$err)
  cat(sprintf("%-13s  mean band %5.2f MeV   rms error %5.2f MeV   inside 1 sigma %3.0f %%   inside 2 sigma %3.0f %%\n",
              nm, mean(s), rms(k$err), 100 * mean(e <= s), 100 * mean(e <= 2 * s)))
}
cat("  (a 1-sigma band should hold about 68 %, a 2-sigma band about 95 %)\n")

k <- blind[["five uniform"]]; s <- band(k)
cat("\nAlmost none of that band comes from the covariance matrix:\n")
cat("  from the constants' covariance:", round(mean(sqrt(rowSums((k$Xt %*% k$fit$C) * k$Xt))) / 1000, 2),
    "MeV      from the fit's own scatter:", round(sqrt(mean(k$fit$r^2)) / 1000, 2), "MeV\n")

cat("\nOutside 2 sigma:", sum(abs(k$err) > 2 * s), "of", nrow(test03), "=",
    round(100 * mean(abs(k$err) > 2 * s)), "%, with errors from",
    round(min(abs(k$err)[abs(k$err) > 2 * s]), 1), "to", round(max(abs(k$err)), 1), "MeV\n")

# The other candidate from D6: the spread of the prediction across the weightings.
P <- sapply(paste("five", names(wts)), function(n) drop(blind[[n]]$Xt %*% blind[[n]]$fit$p))
spread <- (apply(P, 1, max) - apply(P, 1, min)) / 1000
cat("\nSpread across the five weightings: median", round(median(spread), 2),
    "MeV against a median |error| of", round(median(abs(k$err)), 2), "MeV;",
    round(100 * mean(abs(k$err) <= spread / 2)), "% of errors fall inside it.\n")

The 1σ band holds 64 %, which is almost exactly the 68 % it should. The 2σ band holds
83 % instead of 95 %. So the middle of the distribution is well described and **the
tail is not**: 17 % of the test nuclei fall outside two standard deviations, missing
by 6 to 30 MeV. Those are not unlucky draws from the quoted band, they are nuclei the
formula does not describe.

The covariance matrix turns out to be nearly irrelevant to all of this. It
contributes 0.18 MeV of the 2.98 MeV band; the other 2.98 comes from the fit's own
scatter. The careful covariance work of Phase 2 matters for quoting the constants, not
for predicting a mass.

The spread across the five weightings is not an error bar either: its median is
2.28 MeV, wider than the median error, yet it covers only 36 % of the errors, because
the weightings mostly disagree in the same direction at once.

### Where the errors are

In [ ]:
test03$err5 <- blind[["five uniform"]]$err
test03$err6 <- blind[["six uniform"]]$err
test03$region <- cut(test03$Z, c(0, 20, 50, 82, 200),
                     labels = c("Z<=20", "Z 21-50", "Z 51-82", "Z>82"))

nr <- subset(test03, kind == "neutron-rich extension")
t <- aggregate(cbind(err5, err6) ~ out, nr, function(x) round(rms(x), 2))
t$nuclei     <- aggregate(err5 ~ out, nr, length)$err5
t$mean_signed <- aggregate(err5 ~ out, nr, function(x) round(mean(x), 2))$err5
cat("Neutron-rich extensions, by how far past the 2003 edge they sit:\n")
print(t[, c("out", "nuclei", "err5", "err6", "mean_signed")], row.names = FALSE)
cat("(err5, err6 = rms error, MeV; mean_signed = the signed average, five terms)\n")
cat("The rms climbs from 5.3 MeV one step out to 7.9 MeV four steps out, over 173 of\n")
cat("the 182 extensions. The last three rows are 6, 2 and 1 nuclei and carry no weight.\n")
cat("The signed error tells a second story: +1.9 MeV at one step falling through zero\n")
cat("to negative further out, so near the edge the formula under-binds and beyond four\n")
cat("steps it over-binds.\n")

cat("\nBy region and direction, five terms, uniform:\n")
rr <- aggregate(err5 ~ region + kind, test03, function(x) round(rms(x), 2))
rr$nuclei <- aggregate(err5 ~ region + kind, test03, length)$err5
print(rr[order(rr$kind, rr$region), ], row.names = FALSE)
cat("\nThe light neutron-rich corner is where it collapses, which is exactly where\n")
cat("the drip line has actually been reached.\n")

In [ ]:
# The sensitivity check promised in D18: refit without the 14 nuclei AME2020 no
# longer calls measured.
fit03b <- subset(fit03, paste(Z, N) %in% k20)
fb <- fit_lin(design(fit03b$Z, fit03b$A), fit03b$B)
cat("without those 14:", nrow(fit03b), "nuclei, test rms",
    round(rms((test03$B - drop(X5t %*% fb$p)) / 1000), 2), "MeV\n")
cat("with them:       ", nrow(fit03), "nuclei, test rms",
    round(rms(blind[["five uniform"]]$err), 2), "MeV\n")
cat("largest constant shift:", round(max(abs(fb$p - f03$p)) / 1000, 3),
    "MeV. They make no difference.\n")

### The Garvey–Kelson relations on the same test

Phase 4 measured the relations against hold-outs cut from AME2020. This is the real
version of the same question.

In [ ]:
# Everything AME2003 had measured, at every A, is what a person in 2003 had to chain
# from. The test set keeps the A >= 20 cutoff, so the relations and the formulas are
# being asked about the same 326 nuclei.
sig03 <- list(
  T = sqrt(mean(gk_sum(transverse$plus,   transverse$minus,   known03, known03)^2, na.rm = TRUE)),
  L = sqrt(mean(gk_sum(longitudinal$plus, longitudinal$minus, known03, known03)^2, na.rm = TRUE)))
cat("relation scatter inside AME2003: transverse", round(sig03$T), "keV, longitudinal",
    round(sig03$L), "keV   (AME2020 gave 599 and 486)\n")

gk03 <- chain(known03, test03, sig03)
gk03$kind <- test03$kind; gk03$out <- test03$out
gk03$err5 <- test03$err5; gk03$region <- test03$region
cat("\ntest nuclei:", nrow(gk03), "  reached by chaining:", sum(!is.na(gk03$pred)),
    "  unreachable:", sum(is.na(gk03$pred)), "\n")
print(table(unreachable = gk03$kind[is.na(gk03$pred)]))

g <- subset(gk03, !is.na(pred))
cat("\nOn the", nrow(g), "nuclei the relations could reach:\n")
cat("  rms error:      relations", round(rms(g$err), 2), " five-term formula",
    round(rms(g$err5), 2), "MeV\n")
cat("  median |error|: relations", round(median(abs(g$err)), 2), " five-term formula",
    round(median(abs(g$err5)), 2), "MeV\n")
cat("  mean error:     relations", round(mean(g$err), 2),
    "MeV, so they are not biased outward\n")

cat("\nby direction:\n")
t <- aggregate(cbind(err, err5) ~ kind, g, function(x) round(rms(x), 2))
t$nuclei <- aggregate(err ~ kind, g, length)$err
t$gk_median <- aggregate(abs(err) ~ kind, g, function(x) round(median(x), 2))[, 2]
print(t[, c("kind", "nuclei", "gk_median", "err", "err5")], row.names = FALSE)

cat("\nby region:\n")
rr <- aggregate(cbind(err, err5) ~ region, g, function(x) round(rms(x), 2))
rr$nuclei <- aggregate(err ~ region, g, length)$err
print(rr, row.names = FALSE)

**0.86 MeV rms and 0.30 MeV median, against 4.37 and 1.89 for the formula** on the
same nuclei: better by a factor of five in the rms and six in the median, and with a
mean error of −0.02 MeV, so no systematic drift outward. The advantage is largest
exactly where the formula is worst — 0.24 MeV against 4.67 for Z > 82.

The cost is coverage. The relations cannot reach 57 of the 326, including all twelve
new-element nuclei, because a chain needs five measured neighbours to start from.

### How well did Phase 4's calibration travel?

In [ ]:
# Phase 4 measured the cost of each chained step on AME2020 hold-outs. This is the
# first time that curve meets data it was not built from (D20).
holdout_2020 <- c(0.27, 0.39, 0.57, 0.69, 0.75, 1.01, 1.35)   # Phase 4, Result 4
gr <- aggregate(err ~ round, g, function(x) round(rms(x), 2))
gr$nuclei <- aggregate(err ~ round, g, length)$err
gr$median <- aggregate(abs(err) ~ round, g, function(x) round(median(x), 2))[, 2]
gr$holdout_2020 <- holdout_2020[gr$round]
gr$power_law <- round(0.131 * gr$round^1.37, 2)
gr$formula <- aggregate(err5 ~ round, g, function(x) round(rms(x), 2))$err5
gr$optimistic_by <- round(gr$err / gr$holdout_2020, 1)
print(gr[, c("round", "nuclei", "median", "err", "holdout_2020", "optimistic_by",
             "power_law", "formula")], row.names = FALSE)
cat("\nerr           rms error measured here, on the real seventeen-year gap\n")
cat("holdout_2020  what the Phase 4 hold-out measured at the same chain length\n")
cat("power_law     the fitted curve, 0.13 x rounds^1.37\n")
cat("formula       the five-term formula on the same nuclei\n")

cat("\nThe hold-out was optimistic by a steady factor of about two over the first\n")
cat("three rounds, which is where nearly all of these nuclei sit. Hiding the outer\n")
cat("rim of a finished chart is easier than the real frontier of 2003, which was\n")
cat("ragged, full of gaps, and measured less precisely at its edge.\n")

gband <- 0.131 * g$round^1.37
cat("\nUsing the Phase 4 curve as a 1-sigma band: ",
    round(100 * mean(abs(g$err) <= gband)), "% inside 1 sigma,",
    round(100 * mean(abs(g$err) <= 2 * gband)), "% inside 2 sigma\n")
cat("Doubling it gives", round(100 * mean(abs(g$err) <= 2 * gband)),
    "% inside 1 sigma, which is what a 1-sigma band should hold.\n")
cat("So Phase 7 should carry twice the Phase 4 curve, not the curve.\n")

**The Phase 4 hold-out was optimistic by a factor of about two**, steadily, over the
first three rounds, where 84 % of these nuclei sit. That is a clean result about
hold-out testing in general: cutting the outer rim off a finished chart leaves a
tidy, fully-surrounded edge to predict, while the real 2003 frontier was ragged, full
of gaps, and least precisely measured exactly where the chains had to start.

Used as a 1σ band the Phase 4 curve holds 51 % rather than 68 %. **Doubling it gives
68 %.** So Phase 7 carries twice the Phase 4 curve, which is a correction Phase 5
earned rather than assumed.

### The worst misses

In [ ]:
el <- c("H","He","Li","Be","B","C","N","O","F","Ne","Na","Mg","Al","Si","P","S","Cl",
  "Ar","K","Ca","Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Ga","Ge","As","Se",
  "Br","Kr","Rb","Sr","Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd","In","Sn","Sb",
  "Te","I","Xe","Cs","Ba","La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er",
  "Tm","Yb","Lu","Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg","Tl","Pb","Bi","Po","At",
  "Rn","Fr","Ra","Ac","Th","Pa","U","Np","Pu","Am","Cm","Bk","Cf","Es","Fm","Md","No",
  "Lr","Rf","Db","Sg","Bh","Hs","Mt","Ds","Rg","Cn","Nh","Fl","Mc","Lv","Ts","Og")

ord <- order(-abs(test03$err5))[1:12]
worst <- data.frame(nuclide = paste0(el[test03$Z[ord]], "-", test03$A[ord]),
                    Z = test03$Z[ord], N = test03$N[ord], kind = test03$kind[ord],
                    formula = round(test03$err5[ord], 1),
                    relations = round(gk03$err[ord], 2),
                    rounds = gk03$round[ord])
print(worst, row.names = FALSE)
cat("\nformula, relations = error in MeV; NA = the relations could not reach it.\n")
cat("Every one of the worst twelve is in the light neutron-rich corner, and where the\n")
cat("relations reach at all they are wrong by well under a MeV on nuclei the formula\n")
cat("misses by fifteen.\n")

### Both methods together

In [ ]:
# Chain where a chain exists, formula where it does not - which is what anyone would
# actually do, and which Phase 7 will have to do.
combined <- ifelse(!is.na(gk03$pred), gk03$err, test03$err5)
cat("over all", nrow(test03), "test nuclei:\n")
cat(sprintf("  %-22s rms %5.2f MeV   median |error| %5.2f MeV\n",
            "relations + formula", rms(combined), median(abs(combined))))
cat(sprintf("  %-22s rms %5.2f MeV   median |error| %5.2f MeV\n",
            "five-term formula", rms(test03$err5), median(abs(test03$err5))))
cat(sprintf("  %-22s rms %5.2f MeV   median |error| %5.2f MeV\n",
            "six-term formula", rms(test03$err6), median(abs(test03$err6))))
cat("\nThe median falls by a factor of five. The rms barely moves, because the 57\n")
cat("nuclei the relations cannot reach include the worst ones - boron-21 and\n")
cat("boron-20 are unreachable and the formula misses them by 30 MeV, so they\n")
cat("dominate any sum of squares no matter what is done elsewhere.\n")

par(mfrow = c(1, 2))
plot(test03$N, test03$err5, pch = 20, cex = 0.5, col = "grey40",
     xlab = "Neutron number N", ylab = "Measured minus predicted B (MeV)",
     main = "Five-term formula, blind", ylim = c(-32, 32))
abline(h = 0, col = "red")
plot(gk03$N, gk03$err, pch = 20, cex = 0.5, col = "grey40",
     xlab = "Neutron number N", ylab = "", main = "Garvey-Kelson, blind",
     ylim = c(-32, 32))
abline(h = 0, col = "red")
par(mfrow = c(1, 1))